# 05 — Batch Prediction

Inferencia masiva de clientes usando el pipeline entrenado.

## Objetivos

- Cargar nuevos datos para inferencia.
- Validar estructura y calidad de entrada.
- Generar probabilidades y clases predichas.
- Aplicar umbrales configurables.
- Priorizar clientes para campañas de retención.
- Exportar resultados auditables.
- Medir tiempos y volúmenes de procesamiento.


## 1. Configuración del entorno

Ejecute el notebook desde la raíz del repositorio.

En Google Colab:

```python
!git clone <URL_DEL_REPOSITORIO>
%cd customer-intelligence-ml-platform
```


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (candidate / "src").exists():
            PROJECT_ROOT = candidate
            break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


Project root: /workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1


## 2. Importaciones


In [2]:
import json
import time
import joblib
import numpy as np
import pandas as pd

from src.data import load_customer_data
from src.models import load_model, predict_customers
from src.utils import get_project_path, get_logger

logger = get_logger("notebook.batch_prediction")


## 3. Rutas de entrada y salida


In [3]:
INPUT_PATH = get_project_path(
    "data",
    "raw",
    "customer_churn.csv",
)

MODEL_PATH = get_project_path(
    "artifacts",
    "models",
    "selected_churn_pipeline.joblib",
)

PREDICTIONS_PATH = get_project_path(
    "reports",
    "predictions",
    "customer_predictions.csv",
    create_parent=True,
)

SUMMARY_PATH = get_project_path(
    "reports",
    "metrics",
    "batch_prediction_summary.json",
    create_parent=True,
)

print(INPUT_PATH)
print(MODEL_PATH)
print(PREDICTIONS_PATH)
print(SUMMARY_PATH)


/workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1/data/raw/customer_churn.csv
/workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1/artifacts/models/selected_churn_pipeline.joblib
/workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1/reports/predictions/customer_predictions.csv
/workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1/reports/metrics/batch_prediction_summary.json


## 4. Carga de datos para inferencia


In [4]:
input_df = load_customer_data(
    INPUT_PATH,
    validate=True,
)

print(f"Rows: {len(input_df):,}")
print(f"Columns: {input_df.shape[1]}")
input_df.head()


2026-10-02 03:25:30,666 | INFO | src.data.data_loader | Loaded customer dataset with shape (1000, 21)


Rows: 1,000
Columns: 21


,customer_id,gender,age,region,customer_segment,contract_type,tenure_months,monthly_fee,total_spent,internet_service,...,streaming_service,support_calls,complaints,payment_method,last_payment_delay,digital_usage_score,marketing_score,preferred_contact_hour,internal_campaign_code,churn
0,NT-000001,Masculino,28.0,Lima,Masivo,Anual,40,114.04,3812.50,Fibra,...,No,0,1,Transferencia,2,90.4,69.5,20,CMP-D,0
1,NT-000002,Femenino,50.0,Norte,Masivo,Anual,12,115.97,1468.32,Fibra,...,Sí,1,1,Transferencia,1,63.7,64.2,17,CMP-D,0
2,NT-000003,Masculino,38.0,Lima,Joven digital,Mensual,32,114.42,3128.82,Fibra,...,No,2,0,Tarjeta,5,94.5,63.1,10,CMP-D,1
3,NT-000004,Masculino,34.0,Norte,Joven digital,Anual,54,41.89,2037.76,Sin internet,...,No,1,1,Tarjeta,5,33.3,53.0,12,CMP-C,0
4,NT-000005,Femenino,37.0,Lima,Joven digital,Bianual,9,91.54,790.10,Fibra,...,No,2,1,Tarjeta,2,73.8,85.7,12,CMP-B,0


Aunque el dataset de ejemplo contiene `churn`, en producción la variable objetivo no estará disponible al momento de predecir.


In [5]:
inference_df = input_df.drop(
    columns=["churn"],
    errors="ignore",
).copy()

print(inference_df.shape)
inference_df.head()


(1000, 20)


,customer_id,gender,age,region,customer_segment,contract_type,tenure_months,monthly_fee,total_spent,internet_service,tv_service,streaming_service,support_calls,complaints,payment_method,last_payment_delay,digital_usage_score,marketing_score,preferred_contact_hour,internal_campaign_code
0,NT-000001,Masculino,28.0,Lima,Masivo,Anual,40,114.04,3812.50,Fibra,Sí,No,0,1,Transferencia,2,90.4,69.5,20,CMP-D
1,NT-000002,Femenino,50.0,Norte,Masivo,Anual,12,115.97,1468.32,Fibra,No,Sí,1,1,Transferencia,1,63.7,64.2,17,CMP-D
2,NT-000003,Masculino,38.0,Lima,Joven digital,Mensual,32,114.42,3128.82,Fibra,Sí,No,2,0,Tarjeta,5,94.5,63.1,10,CMP-D
3,NT-000004,Masculino,34.0,Norte,Joven digital,Anual,54,41.89,2037.76,Sin internet,No,No,1,1,Tarjeta,5,33.3,53.0,12,CMP-C
4,NT-000005,Femenino,37.0,Lima,Joven digital,Bianual,9,91.54,790.10,Fibra,No,No,2,1,Tarjeta,2,73.8,85.7,12,CMP-B


## 5. Validación de columnas


In [6]:
required_columns = {
    "customer_id",
    "gender",
    "age",
    "region",
    "customer_segment",
    "contract_type",
    "tenure_months",
    "monthly_fee",
    "total_spent",
    "internet_service",
    "tv_service",
    "streaming_service",
    "support_calls",
    "complaints",
    "payment_method",
    "last_payment_delay",
    "digital_usage_score",
    "marketing_score",
    "preferred_contact_hour",
}

missing_columns = sorted(
    required_columns - set(inference_df.columns)
)

extra_columns = sorted(
    set(inference_df.columns) - required_columns
)

{
    "missing_columns": missing_columns,
    "extra_columns": extra_columns,
}


{'missing_columns': [], 'extra_columns': ['internal_campaign_code']}

In [7]:
if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Input schema validation passed.")


Input schema validation passed.


## 6. Calidad de datos de entrada


In [8]:
quality_summary = pd.DataFrame({
    "dtype": inference_df.dtypes.astype(str),
    "missing_count": inference_df.isna().sum(),
    "missing_pct": (
        inference_df.isna().sum()
        / len(inference_df)
        * 100
    ),
    "unique_values": inference_df.nunique(
        dropna=True
    ),
})

quality_summary.sort_values(
    "missing_pct",
    ascending=False,
).head(15)


,dtype,missing_count,missing_pct,unique_values
region,object,49,4.9,5
digital_usage_score,float64,45,4.5,445
marketing_score,float64,45,4.5,436
total_spent,float64,43,4.3,957
age,float64,41,4.1,55
internet_service,object,33,3.3,3
customer_id,object,0,0.0,1000
gender,object,0,0.0,2
monthly_fee,float64,0,0.0,941
tenure_months,int64,0,0.0,108


In [9]:
duplicate_ids = int(
    inference_df[
        "customer_id"
    ].duplicated().sum()
)

if duplicate_ids > 0:
    raise ValueError(
        f"Found {duplicate_ids} duplicated customer IDs."
    )

print("No duplicated customer IDs.")


No duplicated customer IDs.


## 7. Carga del pipeline


In [10]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        "No se encontró el modelo entrenado. "
        "Ejecute primero 02_model_development.ipynb."
    )

pipeline = load_model(MODEL_PATH)

print(type(pipeline))
print(pipeline.named_steps.keys())


<class 'sklearn.pipeline.Pipeline'>
dict_keys(['preprocessor', 'classifier'])


## 8. Predicción batch con umbral base


In [11]:
DEFAULT_THRESHOLD = 0.50

start_time = time.perf_counter()

prediction_df = predict_customers(
    pipeline,
    inference_df,
    threshold=DEFAULT_THRESHOLD,
)

elapsed_seconds = (
    time.perf_counter()
    - start_time
)

prediction_df.head()


,customer_id,churn_probability,churn_prediction
0,NT-000001,0.130985,0
1,NT-000002,0.386012,0
2,NT-000003,0.713476,1
3,NT-000004,0.171500,0
4,NT-000005,0.222182,0


In [12]:
print(f"Predicted rows: {len(prediction_df):,}")
print(f"Elapsed seconds: {elapsed_seconds:.4f}")
print(
    "Rows per second: "
    f"{len(prediction_df) / elapsed_seconds:,.2f}"
)


Predicted rows: 1,000
Elapsed seconds: 0.1406
Rows per second: 7,112.52


## 9. Integración con atributos de negocio


In [13]:
business_columns = [
    "customer_id",
    "region",
    "customer_segment",
    "contract_type",
    "monthly_fee",
    "tenure_months",
    "support_calls",
    "complaints",
    "last_payment_delay",
]

enriched_predictions = (
    inference_df[business_columns]
    .merge(
        prediction_df,
        on="customer_id",
        how="left",
        validate="one_to_one",
    )
)

enriched_predictions.head()


,customer_id,region,customer_segment,contract_type,monthly_fee,tenure_months,support_calls,complaints,last_payment_delay,churn_probability,churn_prediction
0,NT-000001,Lima,Masivo,Anual,114.04,40,0,1,2,0.130985,0
1,NT-000002,Norte,Masivo,Anual,115.97,12,1,1,1,0.386012,0
2,NT-000003,Lima,Joven digital,Mensual,114.42,32,2,0,5,0.713476,1
3,NT-000004,Norte,Joven digital,Anual,41.89,54,1,1,5,0.171500,0
4,NT-000005,Lima,Joven digital,Bianual,91.54,9,2,1,2,0.222182,0


## 10. Segmentación por nivel de riesgo


In [14]:
def assign_risk_band(
    probability: float,
) -> str:
    if probability >= 0.75:
        return "Muy alto"
    if probability >= 0.55:
        return "Alto"
    if probability >= 0.35:
        return "Medio"
    return "Bajo"

enriched_predictions["risk_band"] = (
    enriched_predictions[
        "churn_probability"
    ].apply(assign_risk_band)
)

enriched_predictions[
    "risk_band"
].value_counts()


risk_band
Bajo        547
Medio       223
Muy alto    144
Alto         86
Name: count, dtype: int64

## 11. Priorización comercial


In [15]:
enriched_predictions[
    "retention_priority"
] = np.select(
    [
        enriched_predictions[
            "churn_probability"
        ] >= 0.75,
        enriched_predictions[
            "churn_probability"
        ] >= 0.55,
        enriched_predictions[
            "churn_probability"
        ] >= 0.35,
    ],
    [
        1,
        2,
        3,
    ],
    default=4,
)

priority_table = (
    enriched_predictions
    .sort_values(
        [
            "retention_priority",
            "churn_probability",
            "monthly_fee",
        ],
        ascending=[
            True,
            False,
            False,
        ],
    )
)

priority_table.head(20)


,customer_id,region,customer_segment,contract_type,monthly_fee,tenure_months,support_calls,complaints,last_payment_delay,churn_probability,churn_prediction,risk_band,retention_priority
451,NT-000452,Norte,Masivo,Mensual,87.58,20,4,2,11,0.911762,1,Muy alto,1
286,NT-000287,Lima,Joven digital,Mensual,63.87,13,4,4,0,0.911517,1,Muy alto,1
320,NT-000321,Sur,Masivo,Mensual,55.40,18,2,2,17,0.909116,1,Muy alto,1
640,NT-000641,Sur,Familia,Mensual,87.83,32,3,0,13,0.908687,1,Muy alto,1
361,NT-000362,Lima,Joven digital,Mensual,53.06,13,6,2,6,0.904506,1,Muy alto,1
241,NT-000242,Norte,Masivo,Mensual,105.18,14,2,1,8,0.899634,1,Muy alto,1
779,NT-000780,Lima,Masivo,Mensual,77.19,13,3,2,3,0.898714,1,Muy alto,1
962,NT-000963,Norte,Joven digital,Mensual,100.59,23,1,1,14,0.895414,1,Muy alto,1
859,NT-000860,Norte,Familia,Mensual,96.34,50,5,4,2,0.894822,1,Muy alto,1
364,NT-000365,Sur,Familia,Mensual,108.77,11,2,1,13,0.886240,1,Muy alto,1


## 12. Cupo operativo de campaña


In [16]:
CAMPAIGN_CAPACITY = 150

campaign_list = (
    priority_table
    .head(CAMPAIGN_CAPACITY)
    .copy()
)

print(
    f"Campaign capacity: {CAMPAIGN_CAPACITY}"
)
print(
    "Average churn probability: "
    f"{campaign_list['churn_probability'].mean():.2%}"
)
print(
    "Estimated monthly revenue at risk: "
    f"S/ {campaign_list['monthly_fee'].sum():,.2f}"
)


Campaign capacity: 150
Average churn probability: 81.43%
Estimated monthly revenue at risk: S/ 15,033.76


## 13. Comparación de umbrales


In [17]:
thresholds = [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
]

threshold_summary = []

probabilities = prediction_df[
    "churn_probability"
].to_numpy()

for threshold in thresholds:
    selected = (
        probabilities >= threshold
    )

    threshold_summary.append({
        "threshold": threshold,
        "selected_customers": int(
            selected.sum()
        ),
        "selected_pct": float(
            selected.mean()
        ),
        "estimated_monthly_fee_at_risk": float(
            inference_df.loc[
                selected,
                "monthly_fee",
            ].sum()
        ),
    })

threshold_summary_df = pd.DataFrame(
    threshold_summary
)

threshold_summary_df


,threshold,selected_customers,selected_pct,estimated_monthly_fee_at_risk
0,0.3,544,0.544,55545.39
1,0.4,370,0.370,37529.94
2,0.5,265,0.265,26787.32
3,0.6,217,0.217,22047.78
4,0.7,177,0.177,17891.91


## 14. Análisis por región


In [18]:
regional_summary = (
    enriched_predictions
    .groupby(
        "region",
        dropna=False,
    )
    .agg(
        customers=(
            "customer_id",
            "count",
        ),
        average_probability=(
            "churn_probability",
            "mean",
        ),
        predicted_churn_rate=(
            "churn_prediction",
            "mean",
        ),
        monthly_fee_at_risk=(
            "monthly_fee",
            lambda series: series[
                enriched_predictions.loc[
                    series.index,
                    "churn_prediction",
                ] == 1
            ].sum(),
        ),
    )
    .sort_values(
        "average_probability",
        ascending=False,
    )
)

regional_summary


,customers,average_probability,predicted_churn_rate,monthly_fee_at_risk
region,,,,
Norte,194,0.428373,0.309278,6359.92
Sur,124,0.393050,0.290323,3777.85
Lima,419,0.385755,0.269690,11019.71
NaN,49,0.381070,0.285714,1307.82
Oriente,89,0.364103,0.224719,2216.25
Centro,125,0.344253,0.176000,2105.77


## 15. Análisis por tipo de contrato


In [19]:
contract_summary = (
    enriched_predictions
    .groupby(
        "contract_type",
        dropna=False,
    )
    .agg(
        customers=(
            "customer_id",
            "count",
        ),
        average_probability=(
            "churn_probability",
            "mean",
        ),
        predicted_churn_rate=(
            "churn_prediction",
            "mean",
        ),
    )
    .sort_values(
        "predicted_churn_rate",
        ascending=False,
    )
)

contract_summary


,customers,average_probability,predicted_churn_rate
contract_type,,,
Mensual,512,0.498195,0.404297
Anual,332,0.297062,0.144578
Bianual,156,0.217194,0.064103


## 16. Validación de probabilidades


In [20]:
probability_checks = {
    "minimum": float(
        prediction_df[
            "churn_probability"
        ].min()
    ),
    "maximum": float(
        prediction_df[
            "churn_probability"
        ].max()
    ),
    "mean": float(
        prediction_df[
            "churn_probability"
        ].mean()
    ),
    "outside_range": int(
        (
            ~prediction_df[
                "churn_probability"
            ].between(0, 1)
        ).sum()
    ),
}

probability_checks


{'minimum': 0.04010314968294001,
 'maximum': 0.9117620971175717,
 'mean': 0.38758298586076384,
 'outside_range': 0}

## 17. Validación de consistencia


In [21]:
consistency_checks = {
    "same_row_count": (
        len(inference_df)
        == len(prediction_df)
    ),
    "unique_output_ids": (
        prediction_df[
            "customer_id"
        ].is_unique
    ),
    "valid_prediction_values": set(
        prediction_df[
            "churn_prediction"
        ].unique()
    ).issubset({0, 1}),
    "probabilities_in_range": (
        prediction_df[
            "churn_probability"
        ].between(0, 1).all()
    ),
}

consistency_checks


{'same_row_count': True,
 'unique_output_ids': True,
 'valid_prediction_values': True,
 'probabilities_in_range': np.True_}

In [22]:
if not all(
    consistency_checks.values()
):
    raise RuntimeError(
        "Batch prediction validation failed."
    )

print("Batch prediction validation passed.")


Batch prediction validation passed.


## 18. Exportación de predicciones


In [23]:
PREDICTIONS_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

priority_table.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(PREDICTIONS_PATH)


/workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1/reports/predictions/customer_predictions.csv


## 19. Exportación de lista de campaña


In [24]:
CAMPAIGN_PATH = get_project_path(
    "reports",
    "predictions",
    "retention_campaign.csv",
    create_parent=True,
)

campaign_list.to_csv(
    CAMPAIGN_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(CAMPAIGN_PATH)


/workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1/reports/predictions/retention_campaign.csv


## 20. Resumen operativo


In [25]:
batch_summary = {
    "input_file": str(
        INPUT_PATH
    ),
    "model_file": str(
        MODEL_PATH
    ),
    "rows_received": int(
        len(inference_df)
    ),
    "rows_predicted": int(
        len(prediction_df)
    ),
    "threshold": float(
        DEFAULT_THRESHOLD
    ),
    "predicted_churn_count": int(
        prediction_df[
            "churn_prediction"
        ].sum()
    ),
    "predicted_churn_rate": float(
        prediction_df[
            "churn_prediction"
        ].mean()
    ),
    "average_churn_probability": float(
        prediction_df[
            "churn_probability"
        ].mean()
    ),
    "processing_seconds": float(
        elapsed_seconds
    ),
    "rows_per_second": float(
        len(prediction_df)
        / elapsed_seconds
    ),
    "campaign_capacity": int(
        CAMPAIGN_CAPACITY
    ),
    "consistency_checks": {
        key: bool(value)
        for key, value in consistency_checks.items()
    },
}

batch_summary


{'input_file': '/workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1/data/raw/customer_churn.csv',
 'model_file': '/workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1/artifacts/models/selected_churn_pipeline.joblib',
 'rows_received': 1000,
 'rows_predicted': 1000,
 'threshold': 0.5,
 'predicted_churn_count': 265,
 'predicted_churn_rate': 0.265,
 'average_churn_probability': 0.38758298586076384,
 'processing_seconds': 0.14059713899996495,
 'rows_per_second': 7112.520262594029,
 'campaign_capacity': 150,
 'consistency_checks': {'same_row_count': True,
  'unique_output_ids': True,
  'valid_prediction_values': True,
  'probabilities_in_range': True}}

## 21. Exportación del resumen


In [26]:
SUMMARY_PATH.write_text(
    json.dumps(
        batch_summary,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(SUMMARY_PATH)


/workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1/reports/metrics/batch_prediction_summary.json


## 22. Lectura de resultados exportados


In [27]:
exported_predictions = pd.read_csv(
    PREDICTIONS_PATH
)

exported_campaign = pd.read_csv(
    CAMPAIGN_PATH
)

print(
    exported_predictions.shape
)
print(
    exported_campaign.shape
)

exported_campaign.head()


(1000, 13)
(150, 13)


,customer_id,region,customer_segment,contract_type,monthly_fee,tenure_months,support_calls,complaints,last_payment_delay,churn_probability,churn_prediction,risk_band,retention_priority
0,NT-000452,Norte,Masivo,Mensual,87.58,20,4,2,11,0.911762,1,Muy alto,1
1,NT-000287,Lima,Joven digital,Mensual,63.87,13,4,4,0,0.911517,1,Muy alto,1
2,NT-000321,Sur,Masivo,Mensual,55.40,18,2,2,17,0.909116,1,Muy alto,1
3,NT-000641,Sur,Familia,Mensual,87.83,32,3,0,13,0.908687,1,Muy alto,1
4,NT-000362,Lima,Joven digital,Mensual,53.06,13,6,2,6,0.904506,1,Muy alto,1


## 23. Simulación de archivo nuevo


In [28]:
SAMPLE_INPUT_PATH = get_project_path(
    "data",
    "external",
    "customer_batch_sample.csv",
    create_parent=True,
)

inference_df.head(50).to_csv(
    SAMPLE_INPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(SAMPLE_INPUT_PATH)


/workspaces/customer-intelligence-ml-platform/entregable_D3_16_repositorio_final_curso_1/data/external/customer_batch_sample.csv


## 24. Predicción sobre archivo externo


In [29]:
external_df = pd.read_csv(
    SAMPLE_INPUT_PATH
)

external_predictions = predict_customers(
    pipeline,
    external_df,
    threshold=0.50,
)

external_predictions.head()


,customer_id,churn_probability,churn_prediction
0,NT-000001,0.130985,0
1,NT-000002,0.386012,0
2,NT-000003,0.713476,1
3,NT-000004,0.171500,0
4,NT-000005,0.222182,0


## 25. Manejo de un archivo defectuoso


In [30]:
broken_external_df = (
    external_df
    .drop(
        columns=[
            "monthly_fee",
        ]
    )
)

try:
    predict_customers(
        pipeline,
        broken_external_df,
    )
except Exception as error:
    print(type(error).__name__)
    print(error)


ValueError
Missing modeling columns: ['monthly_fee']


## 26. Consideraciones para producción

Antes de automatizar batch prediction, se debería incorporar:

- identificación de versión del modelo;
- fecha y hora de inferencia;
- validación formal del esquema;
- logging estructurado;
- almacenamiento del archivo original;
- control de duplicados;
- monitoreo del volumen;
- alertas por datos atípicos;
- trazabilidad entre entrada y salida.


## 27. Preguntas para estudiantes


1. ¿Por qué conviene conservar `customer_id` en la salida?
2. ¿Qué diferencia existe entre probabilidad, clase y banda de riesgo?
3. ¿Cómo elegiría el tamaño de la campaña?
4. ¿Qué ocurriría si se cambiara el umbral a 0.30?
5. ¿Qué validaciones deberían realizarse antes de llamar al modelo?
6. ¿Qué columnas debería contener un archivo batch real?
7. ¿Cómo aseguraría trazabilidad entre entrada y predicción?


## 28. Checklist de cierre


- [ ] Se validó el esquema de entrada.
- [ ] Se verificaron IDs duplicados.
- [ ] Se cargó correctamente el pipeline.
- [ ] Se generaron probabilidades.
- [ ] Se asignaron clases predichas.
- [ ] Se crearon bandas de riesgo.
- [ ] Se priorizó la campaña.
- [ ] Se compararon umbrales.
- [ ] Se analizaron regiones y contratos.
- [ ] Se validó consistencia de salida.
- [ ] Se exportaron predicciones.
- [ ] Se exportó la campaña.
- [ ] Se guardó el resumen operativo.
- [ ] Se probó un archivo externo.
- [ ] Se simuló un error de esquema.


## Resultado esperado

El notebook transforma el pipeline entrenado en un proceso batch reproducible.

La salida principal es una tabla priorizada con:

- identificador del cliente;
- probabilidad de churn;
- clase predicha;
- banda de riesgo;
- prioridad comercial;
- variables relevantes para la campaña.
